<a href="https://colab.research.google.com/github/natchanant-arch/Project_Savings_Cooperative/blob/First/Final_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ธุรกิจสหกรณ์ออมทรัพย์ (Savings Cooperative)

โดยมีระบบบัญชี ฝาก-ถอน-โอน และสมาชิกจะสามารถทำธุรกรรมได้ทีละรายการ (ฝาก/ถอน/โอน) มีการตรวจสอบยอดเงินเพียงพอก่อนทำรายการ คำนวณดอกเบี้ยจากเงินคงเหลือในบัญชีในช่วงสิ้นปี

## Import เพื่อ เรียกใช้งานชุดคำสั่ง หรือฟังก์ชันสำเร็จรูป

In [ ]:
import random
import time
from datetime import datetime, timedelta
!pip install Faker
from faker import Faker
fake = Faker("th_TH")
random.seed(1)
import pandas as pd
import matplotlib, os, shutil
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 25.0 MB/s eta 0:00:00


In [ ]:
# 1. ติดตั้งฟอนต์ภาษาไทย
!apt-get -y install fonts-thai-tlwg

# 2. ล้าง cache ของ matplotlib เพื่ออัปเดตฟอนต์ใหม่
cache_dir = matplotlib.get_cachedir()
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)

# 3. ลงทะเบียนฟอนต์ใหม่เข้ากับ FontManager
font_path = '/usr/share/fonts/truetype/tlwg/Loma.ttf'
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)

# 4. ตั้งค่าฟอนต์หลัก
plt.rcParams['font.family'] = 'Loma'
plt.rcParams['axes.unicode_minus'] = False

print("ตั้งค่าระบบฟอนต์ภาษาไทยสำเร็จ!")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  fonts-tlwg-garuda fonts-tlwg-garuda-ttf fonts-tlwg-kinnari
  fonts-tlwg-kinnari-ttf fonts-tlwg-laksaman fonts-tlwg-laksaman-ttf
  fonts-tlwg-loma fonts-tlwg-loma-ttf fonts-tlwg-mono fonts-tlwg-mono-ttf
  fonts-tlwg-norasi fonts-tlwg-norasi-ttf fonts-tlwg-purisa
  fonts-tlwg-purisa-ttf fonts-tlwg-sawasdee fonts-tlwg-sawasdee-ttf
  fonts-tlwg-typewriter fonts-tlwg-typewriter-ttf fonts-tlwg-typist
  fonts-tlwg-typist-ttf fonts-tlwg-typo fonts-tlwg-typo-ttf fonts-tlwg-umpush
  fonts-tlwg-umpush-ttf fonts-tlwg-waree fonts-tlwg-waree-ttf
The following NEW packages will be installed:
  fonts-thai-tlwg fonts-tlwg-garuda fonts-tlwg-garuda-ttf fonts-tlwg-kinnari
  fonts-tlwg-kinnari-ttf fonts-tlwg-laksaman fonts-tlwg-laksaman-ttf
  fonts-tlwg-loma fonts-tlwg-loma-ttf fonts-tlwg-mono fonts-tlwg-mono-ttf
  fonts-tlwg-norasi fonts-tlwg-norasi-ttf fo

## ส่วนที่ 1 — เตรียม class และฟังก์ชัน

In [ ]:
class Member:
    """ข้อมูลสมาชิกธนาคารออมทรัพย์"""
    def __init__(self, member_id, customer_name, citizen_id="-", phone_number="-"):
        self.member_id = member_id
        self.customer_name = customer_name
        self.citizen_id = citizen_id
        self.phone_number = phone_number

    def get_info(self):
        return f"ลูกค้า ID: {self.member_id} | ชื่อ: {self.customer_name} | เลขบัตรประชาชน: {self.citizen_id} | เบอร์โทร: {self.phone_number}"

In [ ]:
class Account:
    """บัญชีเงินฝากธนาคารออมทรัพย์"""
    def __init__(self, account_number, balance, owner, interest_rate=0.015):
        self.account_number = account_number
        self.balance = float(balance)
        self.owner = owner #ชื่อเจ้าของบัญชี
        self.interest_rate = interest_rate  # ดอกเบี้ย 1.5% ต่อปี ( default )

    # Validation & Operations การตรวจสอบเงื่อนไขละการดำเนินงาน
    def deposit(self, amount):
        """ฝากเงิน: balance = balance + amount"""
        self.balance += amount
        return "ฝากเงินสำเร็จ"

    def withdraw(self, amount):
        """ถอนเงิน: ตรวจสอบ balance >= amount"""
        if self.balance < amount:
            return f"ยอดเงินไม่พอ (มีอยู่ {self.balance:,.2f} บาท)"
        self.balance -= amount
        return "ถอนเงินสำเร็จ"

    def transfer(self, target_account, amount):
        """โอนเงิน: ตัดบัญชีต้นทาง และบวกเข้าบัญชีปลายทาง"""
        if self.balance < amount:
            return f" ยอดเงินไม่พอโอน (มีอยู่ {self.balance:,.2f} บาท)"
        self.balance -= amount
        target_account.balance += amount
        return "โอนเงินสำเร็จ"

    def apply_interest(self):
        """คำนวณดอกเบี้ย: interest = balance * interest_rate แล้วบวกเข้ายอดคงเหลือ"""
        interest = self.balance * self.interest_rate
        self.balance += interest
        return interest



---


มีการแปลงชนิดข้อมูลของยอดเงินให้เป็น Float (จำนวนจริง/ทศนิยม)
และบันทึกเข้าตัวแปรของวัตถุบัญชี เพื่อให้รองรับการคำนวณเศษสตางค์และนำไปบวกลบเงินได้แม่นยำ

---



In [ ]:
import random
from datetime import datetime, timedelta

# 1. ฟังก์ชันสุ่มคิว 40 รายการต่อวัน (รับแค่ txn_id)
def generate_transaction_data(txn_id):
    base_date = datetime(2026, 8, 22)
    day_idx = 0
    total_items = 0

    while True:
        items_today = 40  # กำหนด 40 รายการต่อวันคงที่

        if txn_id <= total_items + items_today:
            queue_num = txn_id - total_items
            current_date = base_date + timedelta(days=day_idx)

            return {
                "วันที่": current_date.strftime("%d/%m/%Y"),
                "หมายเลขคิว": f"A-{queue_num:03d}",
                "queue_seq": queue_num - 1
            }

        total_items += items_today
        day_idx += 1

# 2. Class Transaction
class Transaction:
    def __init__(self, txn_id, account, transaction_type, amount, target_account=None):
        self.txn_id = txn_id

        # 📌 เรียกใช้โดยส่งค่า 40 ตามลำดับตำแหน่งตรงๆ
        date_info = generate_transaction_data(txn_id)
        self.queue_number = date_info["หมายเลขคิว"]
        self.txn_date = date_info["วันที่"]

        # ⏰ คำนวณเวลาเดินหน้าตามลำดับคิวในวันนั้น โดยให้นาทีที่จะบวกเพิ่มเข้าไปจากเวลาเริ่มต้น
        base_start_time = datetime.strptime("08:30:00", "%H:%M:%S")
        queue_seq = date_info["queue_seq"]

        minutes_added = queue_seq * random.randint(8, 11) + random.randint(0, 2) #ลำดับคิวมาคูณกับเวลาสุ่มต่อคิว 8 - 11 นาที และบวกเพิ่มเศษเวลาสุ่มอีกเล็กน้อย 0- 2 นาที
        seconds_added = random.randint(0, 59) # สุ่มเวลาในช่วงวินาทีตั้งแต่ 0 - 59

        actual_time = base_start_time + timedelta(minutes=minutes_added, seconds=seconds_added)
        self.time = actual_time.strftime("%H:%M:%S")

        self.account = account
        self.account_number = account.account_number
        self.customer_name = account.owner.customer_name
        self.transaction_type = transaction_type
        self.amount = amount
        self.target_account = target_account

    def to_dict(self):
        interest = getattr(self.account, "yearly_interest", 0.0)

        if isinstance(self.customer_name, (tuple, list)):
            fname, lname = self.customer_name[0], self.customer_name[1]
        else:
            parts = str(self.customer_name).split(" ", 1)
            fname = parts[0]
            lname = parts[1] if len(parts) > 1 else "-"

        return {
            "ID รายการ": self.txn_id,
            "หมายเลขคิว": self.queue_number,
            "วันที่ทำรายการ": self.txn_date,
            "เวลาทำรายการ": self.time,
            "เลขบัญชี": self.account_number,
            "ชื่อ": fname,
            "นามสกุล": lname,
            "ประเภทรายการ": self.transaction_type,
            "จำนวนเงิน": self.amount,
            "บัญชีปลายทาง": self.target_account if self.target_account else "-",
            "ยอดหลังทำรายการ": round(self.account.balance, 2),
            "ดอกเบี้ยสิ้นปี (1.5%)": round(interest, 2),
            "ยอดรวมดอกเบี้ยสุทธิ": round(self.account.balance + interest, 2)
        }

เป็นการรันวันที่แบบอัตโนมัติ โดยใช้ timedelta เพิ่มจำนวนวันไปเรื่อยๆ เมื่อคิวการทำธุรกรรมในวันนั้นรันจนครบ 40 รายการ

In [ ]:
def generate_thai_name():
    """ฟังก์ชัน: สุ่มชื่อและนามสกุลลูกค้าแยกกัน"""
    name = fake.name()
    first_name, last_name = name.split(" ", 1)

    return f"{first_name} {last_name}"

def random_amount(min_val=100.0, max_val=2000.0):
    """ฟังก์ชัน: สุ่มยอดเงิน -> คืนค่าเป็น float """
    return round(random.uniform(min_val, max_val), 2)

def format_currency(amount, symbol="บาท"):
    """ฟังก์ชัน: จัดรูปแบบตัวเลขเป็นสตริงราคา -> คืนค่าเป็น string"""
    return f"{amount:,.2f} {symbol}"